## 21. 🆕 Backtest con proyecciones antiguas (ONU 2012-2019 e INDEC 2013)

**Pregunta:** ¿cuánto se equivocaron las proyecciones de **fecundidad / nacimientos** y de **población de 65+** que existían *antes* de 2020? Es el único "fuera de muestra" legítimo del proyecto: no se valida el índice (no tiene resultado observable), sino sus **insumos** $N$ y $J$.

**Datos (descargados con autorización de la usuaria, fuentes oficiales):** paquetes R de CRAN con las tablas de la ONU (`wpp2012`, `wpp2015`, `wpp2017`, `wpp2019`) y la *Serie Análisis Demográfico n.º 35* de INDEC (2013). El repositorio guarda solo las filas de Argentina (`vintages_wpp_argentina.csv`).

> ⚠️ Los nacimientos "derivados" son una reconstrucción propia (TGF × patrón de fecundidad por edad × mujeres de cada edad), no un dato de la ONU. Se verifica que, en 2015-2020 (sin shock), el error de la reconstrucción es de −2 % a +7 %.

### 21.1 Módulo (copia de `src/tbp_vintages.py`)

In [ ]:
# ====== MÓDULO tbp_vintages (misma versión que src/tbp_vintages.py) ======
# -*- coding: utf-8 -*-
"""
tbp_vintages.py — Backtest de proyecciones ANTIGUAS (ONU WPP 2012/2015/2017/2019 e INDEC 2013) contra lo ocurrido.

Pregunta: ¿cuánto se equivocaron las proyecciones de fecundidad/nacimientos y de población de 65+ que estaban
disponibles antes de 2020? Es el único "fuera de muestra" legítimo del proyecto: se valida el INSUMO (N, J), no el índice.

Datos de entrada (todos en el repositorio):
  datos/procesados/vintages_wpp_argentina.csv   filas de Argentina extraídas de los paquetes R `wpp2012/2015/2017/2019`
                                                (CRAN; datos de la División de Población de la ONU)
  datos/procesados/series_tbp_extraidas.csv     nacimientos DEIS y población 65+ ONU 2024 (referencia)
  datos/procesados/renaper_natalidad_2012_2025.csv   TGF observada (RENAPER)
INDEC 2013 (Serie Análisis Demográfico n.º 35, Cuadros 2 y 6) está transcripta abajo con su página.

AVISOS:
  * Los nacimientos "derivados" = TGF × Σ_a (porcentaje de fecundidad por edad_a / 5) × mujeres_a, con las mujeres de
    cada grupo quinquenal promediadas entre el inicio y el fin del período. Es una reconstrucción propia, no un dato de la ONU.
    Control: en 2015-2020, sin shock, el error de esa reconstrucción es de −2 % a +7 %.
  * La "población 65+ observada" usa WPP 2024 como referencia (no es un censo por edad). Para 2025 es proyección.
  * El período 2020-2025 se compara con 2020-2024 observado; con 2025 (provisorio) la TGF media baja de 1,39 a 1,34.
"""
import os, io, urllib.request
import numpy as np
import pandas as pd

RAW_BASE_VINT = "https://raw.githubusercontent.com/carolinamsfelipe/Tp-Indicadores/main/datos/procesados"
RUTAS_VINT = ["../datos/procesados", "datos/procesados", "github_staging/datos/procesados", "datos_tbp", "."]
EDADES_MUJERES = ["15-19", "20-24", "25-29", "30-34", "35-39", "40-44", "45-49"]
LANZAMIENTO = {"wpp2012": 2012, "wpp2015": 2015, "wpp2017": 2017, "wpp2019": 2019}

# INDEC 2013 (Serie 35). Cuadro 6 (p. 17): TGF. Cuadro 2 (pp. 29-30): población de 65+ = suma de 65-69 ... 100 y más, ambos sexos.
INDEC_2013_TGF = {2010: 2.41, 2015: 2.28, 2020: 2.18, 2025: 2.10, 2030: 2.05, 2035: 2.01, 2040: 1.98}
INDEC_2013_J65 = {2018: 4982425, 2019: 5103968, 2020: 5227722, 2021: 5353272, 2022: 5480183, 2023: 5608402, 2024: 5737818, 2025: 5868496}
# INDEC 2025 (proyección base Censo 2022; pp. 39-42) para contraste de 65+
INDEC_2025_J65 = {2022: 5529167, 2023: 5610238, 2024: 5711979, 2025: 5815927}


def _leer_vint(nombre, rutas=None):
    for r in (rutas or RUTAS_VINT):
        p = os.path.join(r, nombre)
        if os.path.exists(p):
            return pd.read_csv(p)
    with urllib.request.urlopen(f"{RAW_BASE_VINT}/{nombre}") as f:
        return pd.read_csv(io.BytesIO(f.read()))


def cargar_entradas_vint(rutas=None):
    v = _leer_vint("vintages_wpp_argentina.csv", rutas)
    ser = _leer_vint("series_tbp_extraidas.csv", rutas).set_index("year")
    ren = _leer_vint("renaper_natalidad_2012_2025.csv", rutas).set_index("anio")
    return v, ser, ren


def _edad_inicio_vint(a):
    return int(str(a).replace("+", "-").split("-")[0])


def _pop_vint(v, vint, anio, edades, sexo_tablas):
    tot = 0.0
    for tab in sexo_tablas:
        d = v[(v.vintage == vint) & (v.tabla == tab) & (v.periodo.astype(str) == str(anio)) & (v.edad.isin(edades))]
        tot += d.valor.sum()
    return tot


def backtest_wpp(v, ser, ren):
    """Compara TGF, nacimientos (derivados) y población 65+ proyectados por cada edición de la ONU con lo observado."""
    tfr_obs = ren.tgf_hijos_por_mujer
    obs_tfr = {"2015-2020": tfr_obs.loc[2015:2019].mean(), "2020-2025": tfr_obs.loc[2020:2024].mean()}
    nac = ser.deis_nacimientos
    obs_b = {"2015-2020": nac.loc[2015:2019].mean(), "2020-2025": nac.loc[2020:2024].mean()}
    filas = []
    for vint, yr in LANZAMIENTO.items():
        for per in ("2015-2020", "2020-2025"):
            t = v[(v.vintage == vint) & (v.tabla == "tfrprojMed") & (v.periodo == per)]
            if t.empty:
                continue
            T = float(t.valor.iloc[0]); y0, y1 = int(per[:4]), int(per[5:])
            pct = np.array([float(v[(v.vintage == vint) & (v.tabla == "percentASFR") & (v.periodo == per) & (v.edad == a)].valor.iloc[0])
                            for a in EDADES_MUJERES]) / 100
            W = []
            for a in EDADES_MUJERES:
                w0 = _pop_vint(v, vint, y0, [a], ["popF", "popFprojMed"]); w1 = _pop_vint(v, vint, y1, [a], ["popF", "popFprojMed"])
                W.append((w0 + w1) / 2)
            births = float((T * pct / 5 * np.array(W)).sum()) * 1000
            hz = (y0 + y1) / 2 - yr
            filas.append((vint, per, "TGF (hijos por mujer)", T, obs_tfr[per], hz))
            filas.append((vint, per, "Nacimientos por año (derivado)", births, obs_b[per], hz))
        edades65 = sorted({a for a in v[(v.vintage == vint) & v.tabla.isin(["popF", "popFprojMed"])].edad.unique() if _edad_inicio_vint(a) >= 65})
        for Y in (2020, 2025):
            p65 = _pop_vint(v, vint, Y, edades65, ["popF", "popFprojMed"]) + _pop_vint(v, vint, Y, edades65, ["popM", "popMprojMed"])
            if p65 > 0:
                filas.append((vint, str(Y), "Población de 65+ (miles)", p65, ser.un_pob_65mas_miles[Y], Y - yr))
    df = pd.DataFrame(filas, columns=["edicion", "periodo", "variable", "proyectado", "observado", "horizonte_anios"])
    df["error_pct"] = (df.proyectado / df.observado - 1) * 100
    return df


def backtest_indec2013(ser, ren):
    tfr = ren.tgf_hijos_por_mujer
    filas = []
    for y in (2015, 2020, 2025):
        filas.append(("INDEC 2013", str(y), "TGF (hijos por mujer)", INDEC_2013_TGF[y], float(tfr[y]), y - 2013, "2025 provisorio" if y == 2025 else ""))
    for y in (2022, 2023, 2024, 2025):
        filas.append(("INDEC 2013", str(y), "Población de 65+ (contra INDEC 2025)", INDEC_2013_J65[y], INDEC_2025_J65[y], y - 2013, ""))
    df = pd.DataFrame(filas, columns=["edicion", "periodo", "variable", "proyectado", "observado", "horizonte_anios", "nota"])
    df["error_pct"] = (df.proyectado / df.observado - 1) * 100
    return df


def traduccion_a_tbp(tbp_c_base, errores_N=(0.17, 0.25, 0.40)):
    """TBP_C es PROPORCIONAL a N: si N de una cohorte futura se equivocara ±x %, el TBP_C cambia ±x % (resto constante)."""
    return pd.DataFrame({"error_en_N": [f"±{int(e*100)} %" for e in errores_N],
                         "TBP_C_bajo": [tbp_c_base * (1 - e) for e in errores_N],
                         "TBP_C_alto": [tbp_c_base * (1 + e) for e in errores_N]})


def autotest_vintages(verbose=True):
    v, ser, ren = cargar_entradas_vint()
    b = backtest_wpp(v, ser, ren)
    # 1) la reconstrucción de nacimientos es razonable cuando NO hubo shock (2015-2020): error entre −5 % y +10 %
    n1 = b[(b.variable.str.startswith("Nacimientos")) & (b.periodo == "2015-2020")]
    assert n1.error_pct.between(-5, 10).all(), n1
    # 2) en 2020-2025 todas las ediciones sobreestimaron la fecundidad por más de 40 %
    t2 = b[(b.variable.str.startswith("TGF")) & (b.periodo == "2020-2025")]
    assert len(t2) == 4 and (t2.error_pct > 40).all(), t2
    # 3) la población de 65+ tiene error pequeño (< 4 %) en 2020
    j20 = b[(b.variable.str.startswith("Población")) & (b.periodo == "2020")]
    assert j20.error_pct.abs().max() < 4, j20
    # 4) el error de fecundidad NO crece de forma sistemática con el horizonte (de 3,5 a 10,5 años va de +48 % a +58 %)
    assert t2.error_pct.max() - t2.error_pct.min() < 15
    i = backtest_indec2013(ser, ren)
    assert abs(i[(i.variable.str.startswith("TGF")) & (i.periodo == "2020")].error_pct.iloc[0] - 33.7) < 1.0
    assert traduccion_a_tbp(0.2, (0.1,)).TBP_C_alto.iloc[0] == 0.2 * 1.1
    if verbose:
        print("Autotest tbp_vintages OK: reconstrucción de nacimientos, errores de TGF, 65+ e INDEC 2013.")
    return b

### 21.2 Proyectado contra observado

In [ ]:
V21, SER21, REN21 = cargar_entradas_vint()
BT21 = backtest_wpp(V21, SER21, REN21)
IN21 = backtest_indec2013(SER21, REN21)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
display(BT21.round(2))
display(IN21.round(2))
t = BT21[(BT21.variable.str.startswith("TGF")) & (BT21.periodo == "2020-2025")]
print("TGF 2020-2025: proyectada", t.proyectado.round(2).tolist(), "| observada (2020-24) =", round(float(t.observado.iloc[0]), 2),
      "| error:", t.error_pct.round(0).astype(int).tolist(), "% (ediciones 2012, 2015, 2017, 2019)")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.4))
t = BT21[(BT21.variable.str.startswith("TGF")) & (BT21.periodo == "2020-2025")]
x = np.arange(len(t) + 1)
vals = list(t.proyectado) + [INDEC_2013_TGF[2020]]
ax[0].bar(x, vals, color=["#4c72b0"] * len(t) + ["#55a868"], alpha=.85)
ax[0].axhline(float(t.observado.iloc[0]), color="#c44e52", lw=2, label="observada 2020-24 (RENAPER) = %.2f" % float(t.observado.iloc[0]))
ax[0].set_xticks(x); ax[0].set_xticklabels([e.replace("wpp", "ONU ") for e in t.edicion] + ["INDEC 2013\n(año 2020)"], fontsize=8)
ax[0].set_ylim(0, 2.6); ax[0].set_title("Fecundidad 2020-2025: proyectada vs observada"); ax[0].set_ylabel("hijos por mujer"); ax[0].legend(fontsize=8)
n = BT21[(BT21.variable.str.startswith("Nacimientos")) & (BT21.periodo == "2020-2025")]
ax[1].bar(range(len(n)), n.proyectado / 1e3, color="#4c72b0", alpha=.85)
ax[1].axhline(float(n.observado.iloc[0]) / 1e3, color="#c44e52", lw=2, label="observados 2020-24 (DEIS)")
ax[1].set_xticks(range(len(n))); ax[1].set_xticklabels([e.replace("wpp", "ONU ") for e in n.edicion]); ax[1].set_title("Nacimientos por año 2020-2025 (derivados)")
ax[1].set_ylabel("miles"); ax[1].legend(fontsize=8)
j = BT21[(BT21.variable.str.startswith("Población")) & (BT21.periodo == "2020")]
ax[2].bar(range(len(j)), j.error_pct, color="#8172b2", alpha=.85); ax[2].axhline(0, color="k", lw=.8)
ax[2].set_xticks(range(len(j))); ax[2].set_xticklabels([e.replace("wpp", "ONU ") for e in j.edicion])
ax[2].set_title("Error de la población de 65+ en 2020 (%)"); ax[2].set_ylim(-12, 12)
plt.tight_layout(); os.makedirs(OUT, exist_ok=True); plt.savefig(os.path.join(OUT, "seccion_21_backtest_vintages.png"), dpi=120); plt.show()

### 21.3 Qué muestra

* **Fecundidad:** todas las ediciones de la ONU (2012, 2015, 2017, 2019) proyectaron para 2020-2025 una TGF de **2,06 a 2,20** hijos por mujer; lo observado fue **~1,39**. Error de **+48 % a +58 %**. INDEC 2013 proyectaba 2,18 para 2020 (observada 1,63) y 2,10 para 2025 (observada 1,04, provisoria).
* **Nacimientos (derivados):** **+38 % a +54 %** por encima de los observados. El error **no se achica** al acercarse el horizonte (la edición 2019, a 3-4 años, falló lo mismo que la de 2012, a 10): fue un **cambio de régimen** que nadie anticipó, no un error que crezca de a poco.
* **Población de 65+ ($J$):** error de **−2,8 % a +0,2 %** en 2020 y de **−1,1 % a +2,6 %** en 2025 (contra WPP 2024). INDEC 2013 contra INDEC 2025: de −0,9 % a +0,9 %. Las personas de 65+ de 2090 **ya nacieron**: su proyección depende de mortalidad y migración, mucho más estables.
* **Conclusión:** el insumo frágil es **$N$ (natalidad) hacia adelante**; $J$ es robusto. Esto refuerza que, para cohortes anteriores a 2025 (con $N$ observado), el indicador es sólido, y que para cohortes futuras la incertidumbre dominante es la natalidad.

### 21.4 Qué significa para el TBP de las cohortes futuras

In [ ]:
def _cargar_modelo_avanzado():
    for r in ["../datos/procesados", "datos/procesados", "github_staging/datos/procesados", "datos_tbp", "."]:
        p = os.path.join(r, "tbp_modelo_avanzado_1950_2035.csv")
        if os.path.exists(p):
            return pd.read_csv(p).set_index("cohorte")
    raise FileNotFoundError("tbp_modelo_avanzado_1950_2035.csv")
MA21 = _cargar_modelo_avanzado()
for c in (2030, 2035):
    base = float(MA21.loc[c, "TBP_C_p4_persistencia"])
    print(f"Cohorte {c} (persistencia): TBP_C = {base:.3f} años. Si N se equivocara (el resto constante):")
    display(traduccion_a_tbp(base, (0.17, 0.25, 0.40)).round(3))
print("TBP_C es proporcional a N: un error de x % en N es un error de x % en el indicador.")

**Lectura.** La banda del Monte Carlo de la sección 19 usa un error de $N$ calibrado con RENAPER/DEIS (σ ≈ 2 %) para cohortes observadas y los **tres escenarios** para las futuras. El backtest muestra que, en el último ciclo, las proyecciones oficiales sobrestimaron los nacimientos entre 38 % y 54 %. **No** significa que los escenarios actuales vayan a fallar así (ahora están anclados en la fecundidad observada de 2025), pero sí que **±17 % es un piso, no un techo**, para la incertidumbre de la natalidad de cohortes posteriores a 2025. Por eso la tabla de arriba muestra también ±25 % y ±40 %.

In [ ]:
_bt = autotest_vintages()

### 21.5 Límites

* Los nacimientos derivados son una reconstrucción propia (error de −2 % a +7 % en 2015-2020, sin shock).
* La "población de 65+ observada" es WPP 2024 (no un censo por edad); para 2025 es proyección.
* El período 2020-2025 se compara con 2020-2024 observado; incluyendo 2025 (provisorio) la TGF media observada baja de 1,39 a 1,34 y los errores suben.
* Cuatro ediciones de la ONU y una de INDEC son una muestra pequeña: no alcanza para estimar cómo crece el error con el horizonte.
* No se encontró la edición WPP 2010; las URL históricas del sitio de la ONU devuelven 404.